## Riemann fixed - Varying features (Fig. 3)

In [5]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def read_metrics_json(file_path, keys):
    with open(file_path, 'r') as f:
        data = json.load(f)

    metrics = ['accuracy', 'balanced_accuracy', 'f1_macro']
    results = {}

    for feature, subkeys in keys.items():
        results[feature] = {}
        for split, key in subkeys.items():
            results[feature][split] = {}
            for metric in metrics:
                values = [entry[metric] for entry in data[key]]
                mean = float(np.mean(values))
                std = float(np.std(values))
                results[feature][split][metric] = {
                    'values': values,
                    'mean': mean,
                    'std': std
                }

    return results

# Define keys for each feature combination
COV_ATM_keys = {
    'ensemble': {
        'train': 'train_metrics_ensemble',
        'validation': 'validation_metrics_ensemble'
    },
    'covariance_matrices': {
        'train': 'train_metrics_covariance_matrices',
        'validation': 'validation_metrics_covariance_matrices'
    },
    'atms': {
        'train': 'train_metrics_atms',
        'validation': 'validation_metrics_atms'
    }
}

COV_CC_keys = {
    'ensemble': {
        'train': 'train_metrics_ensemble',
        'validation': 'validation_metrics_ensemble'
    },
    'covariance_matrices': {
        'train': 'train_metrics_covariance_matrices',
        'validation': 'validation_metrics_covariance_matrices'
    },
    'correlation_matrices': {
        'train': 'train_metrics_correlation_matrices',
        'validation': 'validation_metrics_correlation_matrices'
    }
}

PSD_keys = {
    'ensemble': {
        'train': 'train_metrics_ensemble',
        'validation': 'validation_metrics_ensemble'
    },
    'psd_1': { # LDA or XGboost
        'train': 'train_metrics_psd_1',
        'validation': 'validation_metrics_psd_1'
    },
    'psd_2': { #SVM or Xgoost
        'train': 'train_metrics_psd_2',
        'validation': 'validation_metrics_psd_2'
    }
}


# Riemann
riemann_cov_atm_path = '../Results/num_nodes_50/logs/TSClassifier/metrics.json'
riemann_cov_cc_path = '../Results/num_nodes_50/logs/Corr_Mat/TSClassifier/metrics.json'

riemann_cov_atm_metrics_data = read_metrics_json(riemann_cov_atm_path, COV_ATM_keys)
riemann_cov_cc_metrics_data = read_metrics_json(riemann_cov_cc_path, COV_CC_keys)

atm_riemann = riemann_cov_atm_metrics_data["atms"]["validation"]["balanced_accuracy"]["values"]
cov_riemann = riemann_cov_atm_metrics_data["covariance_matrices"]["validation"]["balanced_accuracy"]["values"]
cor_riemann = riemann_cov_cc_metrics_data["correlation_matrices"]["validation"]["balanced_accuracy"]["values"]
cov_plus_corr = riemann_cov_cc_metrics_data["ensemble"]["validation"]["balanced_accuracy"]["values"]

In [6]:
from scipy.stats import mannwhitneyu


cov_vs_cor = mannwhitneyu(cor_riemann, cov_riemann, alternative="less")
cov_vs_atm = mannwhitneyu(atm_riemann,cov_riemann,alternative="less")
cor_vs_atm = mannwhitneyu(atm_riemann,cor_riemann,alternative="less")
print("cov_vs_cor", cov_vs_cor.pvalue, "\ncov_vs_atm", cov_vs_atm.pvalue, "\ncor_vs_atm", cor_vs_atm.pvalue)


cov_plus_corr_vs_cov = mannwhitneyu(cov_riemann, cov_plus_corr, alternative="less")
print("\n\nCOV+CORR vs COV", cov_plus_corr_vs_cov.pvalue)




cov_vs_cor 0.015873015873015872 
cov_vs_atm 0.007936507936507936 
cor_vs_atm 0.42063492063492064


COV+CORR vs COV 0.04634585806222321


In [ ]:
cov_plus_corr_vs_atm =  mannwhitneyu(atm_riemann, cov_plus_corr, alternative="less")
cov_plus_corr_vs_corr = mannwhitneyu(cor_riemann, cov_plus_corr, alternative="less")

print("COV+CORR vs ATM", cov_plus_corr_vs_atm.pvalue, "\nCOV+CORR vs CORR", cov_plus_corr_vs_corr.pvalue,)

COV+CORR vs ATM 0.003968253968253968 
COV+CORR vs CORR 0.003968253968253968


## Varying Classifier (Fig. 4)

In [7]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import PathPatch

metric = "balanced_accuracy"

"Results/num_nodes_50"
CLASSICAL_MODELS = {
    "LDA": {
        "PSD": ("../Results/logs/PSD/LinearDiscriminantAnalysis/metrics.json", "psd_1"),
        "Ensemble COV+CORR": ("../Results/num_nodes_50/logs/Corr_Mat/LinearDiscriminantAnalysis/metrics.json", "ensemble"),
    },
    "SVM": {
        "PSD": ("../Results/logs/PSD/LinearDiscriminantAnalysis/metrics.json", "psd_2"),
        "Ensemble COV+CORR": ("../Results/num_nodes_50/logs/Corr_Mat/SVC/metrics.json", "ensemble"),
    },
    "XGBoost": {
        "PSD": ("../Results/logs/PSD/XGBClassifier/metrics.json", "psd_1"),
        "Ensemble COV+CORR": ("../Results/num_nodes_50/logs/Corr_Mat/XGBClassifier/metrics.json", "ensemble"),
    }
}

# Riemann paths
RiemANN_MODEL = {
    "Riemann": {
    "Ensemble COV+CORR": ('../Results/num_nodes_50/logs/Corr_Mat/TSClassifier/metrics.json', "ensemble"),
    }
}



# NN paths
NN_BASE =  "../Results/num_nodes_50/NN"  # TODO "../v3_Gio/outputs/" or "../v2/src/scripts/NN/results/logs" 
#NN_file_name = "cv_balanced_accuracy_val.npy"   # TODO "cv_balanced_accuracy_val.npy" or "cv_scores.npy" 

if NN_BASE.split("/")[-1]=="logs":
    NN_file_name = "cv_scores.npy" 
elif metric == "balanced_accuracy":
    NN_file_name = "cv_balanced_accuracy_val.npy"
else:
    NN_file_name = "cv_f1_val.npy"

NN_FEATURES = {
    "PSD": "PSDs",
    "Ensemble COV+CORR": "COV+CORR"
}
NN_MODELS = ["SimpleNN", "DeepNN"]



# ------------------------------------------------------------------
# Utilities
# ------------------------------------------------------------------
def read_metrics_json(path, key):
    with open(path, "r") as f:
        data = json.load(f)
    return [entry[metric] for entry in data[key.replace("ensemble", "validation_metrics_ensemble")]]

# ------------------------------------------------------------------
# Collect data
# ------------------------------------------------------------------
rows = []

# Classical models
for model, feats in CLASSICAL_MODELS.items():
    for feature, (path, key) in feats.items():
        with open(path, "r") as f:
            data = json.load(f)
        vals = [x[metric] for x in data[f"validation_metrics_{key}"]]
        for v in vals:
            rows.append({
                "Model": model,
                "Feature": feature,
                "Score": v
            })

# Neural networks
for model in NN_MODELS:
    for feature, folder in NN_FEATURES.items():
        path = os.path.join(NN_BASE, folder, model, NN_file_name)
        uu = path
        if os.path.exists(path):
            scores = np.load(path)
            for v in scores:
                rows.append({
                    "Model": model,
                    "Feature": feature,
                    "Score": v
                })

# Riemann
for model, feats in RiemANN_MODEL.items():
    for feature, (path, key) in feats.items():
        with open(path, "r") as f:
            data = json.load(f)
        vals = [x[metric] for x in data[f"validation_metrics_{key}"]]
        for v in vals:
            rows.append({
                "Model": model,
                "Feature": feature,
                "Score": v
            })
df = pd.DataFrame(rows)
df

,Model,Feature,Score
0,LDA,PSD,0.331845
1,LDA,PSD,0.187500
2,LDA,PSD,0.333333
3,LDA,PSD,0.187500
4,LDA,PSD,0.333333
5,LDA,Ensemble COV+CORR,0.441964
6,LDA,Ensemble COV+CORR,0.312500
7,LDA,Ensemble COV+CORR,0.333333
8,LDA,Ensemble COV+CORR,0.333333
9,LDA,Ensemble COV+CORR,0.446429


In [8]:
set(df["Model"])

{'DeepNN', 'LDA', 'Riemann', 'SVM', 'SimpleNN', 'XGBoost'}

In [9]:
Riemann_cov_corr = df[(df["Model"]=="Riemann") * (df["Feature"]=="Ensemble COV+CORR")].Score.values

LDA_PSD = df[(df["Model"]=="LDA") * (df["Feature"]=="PSD")].Score.values
SVM_PSD = df[(df["Model"]=="SVM") * (df["Feature"]=="PSD")].Score.values
XGBoost_PSD = df[(df["Model"]=="XGBoost") * (df["Feature"]=="PSD")].Score.values
SimpleNN_PSD = df[(df["Model"]=="SimpleNN") * (df["Feature"]=="PSD")].Score.values
DeepNN_PSD = df[(df["Model"]=="DeepNN") * (df["Feature"]=="PSD")].Score.values

LDA_cov_corr = df[(df["Model"]=="LDA") * (df["Feature"]=="Ensemble COV+CORR")].Score.values
SVM_cov_corr = df[(df["Model"]=="SVM") * (df["Feature"]=="Ensemble COV+CORR")].Score.values
XGBoost_cov_corr = df[(df["Model"]=="XGBoost") * (df["Feature"]=="Ensemble COV+CORR")].Score.values
SimpleNN_cov_corr = df[(df["Model"]=="SimpleNN") * (df["Feature"]=="Ensemble COV+CORR")].Score.values
DeepNN_cov_corr = df[(df["Model"]=="DeepNN") * (df["Feature"]=="Ensemble COV+CORR")].Score.values

In [23]:
alternative = 'less' #'two-sided', "less"
LDA_PSD_RIE = mannwhitneyu(LDA_PSD, Riemann_cov_corr, alternative=alternative)
SVM_PSD_RIE = mannwhitneyu(SVM_PSD, Riemann_cov_corr, alternative=alternative)
XGb_PSD_RIE = mannwhitneyu(XGBoost_PSD, Riemann_cov_corr, alternative=alternative)
SNN_PSD_RIE = mannwhitneyu(SimpleNN_PSD, Riemann_cov_corr, alternative=alternative)
DNN_PSD_RIE = mannwhitneyu(DeepNN_PSD, Riemann_cov_corr, alternative=alternative)

LDA_cov_cor_RIE = mannwhitneyu(LDA_cov_corr, Riemann_cov_corr, alternative=alternative)
SVM_cov_cor_RIE = mannwhitneyu(SVM_cov_corr, Riemann_cov_corr, alternative=alternative)
XGb_cov_cor_RIE = mannwhitneyu(XGBoost_cov_corr, Riemann_cov_corr, alternative=alternative)
SNN_cov_cor_RIE = mannwhitneyu(SimpleNN_cov_corr, Riemann_cov_corr, alternative=alternative)
DNN_cov_cor_RIE = mannwhitneyu(DeepNN_cov_corr, Riemann_cov_corr, alternative=alternative )

In [13]:
LDA_PSD


array([0.33184524, 0.1875    , 0.33333333, 0.1875    , 0.33333333])

In [12]:
SVM_PSD

array([0.33184524, 0.1875    , 0.33333333, 0.1875    , 0.33333333])

In [14]:
Riemann_cov_corr

array([0.71428571, 0.75      , 0.83333333, 0.70833333, 0.77678571])

In [24]:

print("LDA_PSD_RIE:", LDA_PSD_RIE.pvalue)
print("SVM_PSD_RIE:", SVM_PSD_RIE.pvalue)
print("XGb_PSD_RIE:", XGb_PSD_RIE.pvalue)
print("SNN_PSD_RIE:", SNN_PSD_RIE.pvalue)
print("DNN_PSD_RIE:", DNN_PSD_RIE.pvalue)

print("LDA_cov_cor_RIE:", LDA_cov_cor_RIE.pvalue)
print("SVM_cov_cor_RIE:", SVM_cov_cor_RIE.pvalue)
print("XGb_cov_cor_RIE:", XGb_cov_cor_RIE.pvalue)
print("SNN_cov_cor_RIE:", SNN_cov_cor_RIE.pvalue)
print("DNN_cov_cor_RIE:", DNN_cov_cor_RIE.pvalue)


LDA_PSD_RIE: 0.005833656171659693
SVM_PSD_RIE: 0.005833656171659693
XGb_PSD_RIE: 0.003968253968253968
SNN_PSD_RIE: 0.003968253968253968
DNN_PSD_RIE: 0.003968253968253968
LDA_cov_cor_RIE: 0.005962616796508801
SVM_cov_cor_RIE: 0.003968253968253968
XGb_cov_cor_RIE: 0.10364992015687063
SNN_cov_cor_RIE: 0.005962616796508801
DNN_cov_cor_RIE: 0.003968253968253968


In [17]:
LDA_PSD


array([0.33184524, 0.1875    , 0.33333333, 0.1875    , 0.33333333])

In [ ]:
Riemann_cov_corr

0.045736601695948925